# Notebook 2 – Modelado y Evaluación de modelos predictivos

En este notebook realizo las transformaciones necesarias para preparar el dataset para el entrenamiento y predicciónes correpondientes. Tras ese trabajo, realizaré la evaluación de los modelos hasta encontrar el que mejor generalice.

In [17]:
# Cargar datos procesados para ML

import pandas as pd
df_ML = pd.read_csv('../data/processed/df_ML.csv')

df_ML.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   int64 
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EnvironmentSatisfaction   1470 non-null   int64 
 9   Gender                    1470 non-null   object
 10  HourlyRate                1470 non-null   int64 
 11  JobInvolvement            1470 non-null   int64 
 12  JobLevel                  1470 non-null   int64 
 13  JobRole                   1470 non-null   object
 14  JobSatisfaction         

In [18]:
# Cargar metadatos del notebook 1

import pandas as pd
import numpy as np
import pickle
import os

try:
    with open('../data/metadata/clasificacion_variables.pkl', 'rb') as f:
        clasificacion = pickle.load(f)
    
    continuas = clasificacion['continuas']
    ordinales = clasificacion['ordinales']
    binarias = clasificacion['binarias']
    categoricas_texto = clasificacion['categoricas_texto']
    proxy_vars = clasificacion['proxy_vars']
    
    print(f"Variables continuas: {len(continuas)}")
    print(f"Variables ordinales: {len(ordinales)}")
    print(f"Variables binarias: {len(binarias)}")
    print(f"Variables categóricas: {len(categoricas_texto)}")
    
except FileNotFoundError:
    print("   ✗ ERROR: No se encontraron los metadatos")
    print("   Volviendo a clasificar variables...")


Variables continuas: 12
Variables ordinales: 11
Variables binarias: 5
Variables categóricas: 6


In [19]:
''' 
El problema se plantea como una clasificación supervisada binaria, donde el objetivo es predecir si un empleado abandonará
la empresa (Attrition = 1) o no (Attrition = 0).
La pérdida de talento supone un fuerte impacto económico y organizativo, por lo que se prioriza la detección temprana de empleados 
en riesgo (recall alto), incluso a costa de reducir la precisión absoluta del modelo (precision).

Los mejores modelos para este tipo de problema de clasificación binaria son LOGISTIC REGRESSION, RANDOM FOREST y XGBOOST.
Para poder hacerlo, primero se transfan las variables categóricas en variables dummy (one-hot encoding).

Todos estos modelos necesitan variables numéricas.
'''

# Ver detalle de cada variable categórica
for i, cat in enumerate(categoricas_texto, 1):
    valores = df_ML[cat].unique()
    n_valores = len(valores)
    distribucion = df_ML[cat].value_counts()
    
    print(f"\n{i}. {cat} ({n_valores} categorías):")
    print(f"   Valores: {list(valores)}")
    
    # Mostrar distribución (solo si no son muchos valores)
    if n_valores <= 10:
        print(f"   Distribución:")
        for valor, count in distribucion.items():
            porcentaje = (count / len(df_ML)) * 100
            print(f"     • {valor}: {count} ({porcentaje:.1f}%)")



1. BusinessTravel (3 categorías):
   Valores: ['Travel_Rarely', 'Travel_Frequently', 'Non-Travel']
   Distribución:
     • Travel_Rarely: 1043 (71.0%)
     • Travel_Frequently: 277 (18.8%)
     • Non-Travel: 150 (10.2%)

2. Department (3 categorías):
   Valores: ['Sales', 'Research & Development', 'Human Resources']
   Distribución:
     • Research & Development: 961 (65.4%)
     • Sales: 446 (30.3%)
     • Human Resources: 63 (4.3%)

3. EducationField (6 categorías):
   Valores: ['Life Sciences', 'Other', 'Medical', 'Marketing', 'Technical Degree', 'Human Resources']
   Distribución:
     • Life Sciences: 606 (41.2%)
     • Medical: 464 (31.6%)
     • Marketing: 159 (10.8%)
     • Technical Degree: 132 (9.0%)
     • Other: 82 (5.6%)
     • Human Resources: 27 (1.8%)

4. Gender (2 categorías):
   Valores: ['Female', 'Male']
   Distribución:
     • Male: 882 (60.0%)
     • Female: 588 (40.0%)

5. JobRole (9 categorías):
   Valores: ['Sales Executive', 'Research Scientist', 'Laboratory 

In [20]:
''' Se utiliza drop='first' eliminando la primera categoría de cada variable para evitar multicolinealidad 
Es lo más conveniente porque se va a utilizar Regresión Logística que necesita evitar multicolinealidad, 
así no se pierde información (si Female=0, implica Male=1), y se reduce el numero de columnas.
'''
# Calcular cuántas columnas nuevas tendría el dataset tras one-hot encoding

columnas_originales = len(df_ML.columns) - 1  # Excluyendo Attrition
columnas_nuevas_estimadas = columnas_originales - len(categoricas_texto)

for cat in categoricas_texto:
    columnas_nuevas_estimadas += (df_ML[cat].nunique() - 1)  # drop='first'

print(f"\n• Columnas originales (sin Attrition): {columnas_originales}")
print(f"• Columnas después de one-hot (drop='first'): ~{columnas_nuevas_estimadas}")
print(f"• Aumento: {columnas_nuevas_estimadas - columnas_originales} columnas nuevas")

print(f"\n• Variables que generan más columnas:")
for cat in categoricas_texto:
    nuevas = df_ML[cat].nunique() - 1
    if nuevas > 2:  # Si genera 3+ columnas nuevas
        print(f"  - {cat}: {df_ML[cat].nunique()} categorías → {nuevas} columnas nuevas")



• Columnas originales (sin Attrition): 34
• Columnas después de one-hot (drop='first'): ~48
• Aumento: 14 columnas nuevas

• Variables que generan más columnas:
  - EducationField: 6 categorías → 5 columnas nuevas
  - JobRole: 9 categorías → 8 columnas nuevas


In [21]:
# Separación de X e y
 
X = df_ML.drop('Attrition', axis=1)
y = df_ML['Attrition']

print(f"X --> {X.shape[1]} columnas")
print(f"y --> {len(y)} valores")

X --> 34 columnas
y --> 1470 valores


In [22]:
# Creación de OneHotEncoder

from sklearn.preprocessing import OneHotEncoder
onehot = OneHotEncoder(drop='first', sparse_output=False, dtype=int)

In [23]:
# Ajustar el OneHotEncoder a las variables categóricas

X_categoricas = X[categoricas_texto]
onehot.fit(X_categoricas)


,categories,'auto'
,drop,'first'
,sparse_output,False
,dtype,<class 'int'>
,handle_unknown,'error'
,min_frequency,None
,max_categories,None
,feature_name_combiner,'concat'


In [24]:
# Nuevas columnas generadas

nuevas_columnas = onehot.get_feature_names_out(categoricas_texto)
print(f"Se crearán {len(nuevas_columnas)} columnas nuevas:")
for i, col in enumerate(nuevas_columnas[:10], 1):
    print(f"  {i}. {col}")

Se crearán 20 columnas nuevas:
  1. BusinessTravel_Travel_Frequently
  2. BusinessTravel_Travel_Rarely
  3. Department_Research & Development
  4. Department_Sales
  5. EducationField_Life Sciences
  6. EducationField_Marketing
  7. EducationField_Medical
  8. EducationField_Other
  9. EducationField_Technical Degree
  10. Gender_Male


In [25]:
# Transformación de las variables categóricas  

X_categoricas_transformado = onehot.transform(X_categoricas)
print(f"Transformado: {X_categoricas_transformado.shape}")


Transformado: (1470, 20)


In [26]:
# Creación del nuevo DataFrame X_final

X_categoricas_df = pd.DataFrame(
    X_categoricas_transformado,
    columns=nuevas_columnas,
    index=X.index  
)
print(f"DataFrame creado: {X_categoricas_df.shape}")

DataFrame creado: (1470, 20)


In [27]:
# En estos momentos X tiene: continuas + ordinales + binarias + categóricas_texto
# y X_categoricas_df tiene: solo las categóricas transformadas

# Falta por COMBINAR:
    # 1. Las NO categóricas de X (continuas, ordinales, binarias)
    # 2. Las categóricas transformadas (X_categoricas_df)

# 1. Ver qué columnas NO son categóricas
columnas_no_categoricas = [col for col in X.columns if col not in categoricas_texto]
print(f"\n1. Columnas NO categóricas ({len(columnas_no_categoricas)}):")
print(columnas_no_categoricas[:5], "...")

# 2. Ver X_categoricas_df
print(f"\n2. X_categoricas_df tiene {X_categoricas_df.shape[1]} columnas")
print("Primeras columnas:")
print(X_categoricas_df.columns[:5].tolist())



1. Columnas NO categóricas (28):
['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction'] ...

2. X_categoricas_df tiene 20 columnas
Primeras columnas:
['BusinessTravel_Travel_Frequently', 'BusinessTravel_Travel_Rarely', 'Department_Research & Development', 'Department_Sales', 'EducationField_Life Sciences']


In [28]:
'''
Después del one-hot encoding:
- 28 columnas numéricas (continuas + ordinales + binarias)
- 20 columnas dummy (de las 6 categóricas)
Total combinado: 48 columnas 

Entonces, el DataFrame final para modelado tendrá 48 columnas:
--> Todas en formato numérico (0/1 o números) y listas para modelos ML
--> Sin texto porque se ha transformado todo a numérico
--> Sin multicolinealidad porque se ha realizado drop='first'
'''

"\nDespués del one-hot encoding:\n- 28 columnas numéricas (continuas + ordinales + binarias)\n- 20 columnas dummy (de las 6 categóricas)\nTotal combinado: 48 columnas \n\nEntonces, el DataFrame final para modelado tendrá 48 columnas:\n--> Todas en formato numérico (0/1 o números) y listas para modelos ML\n--> Sin texto porque se ha transformado todo a numérico\n--> Sin multicolinealidad porque se ha realizado drop='first'\n"

In [29]:
# combinación de columnas numéricas y dummy

# 1. Identificación de las columnas NO categóricas en X
columnas_no_categoricas = [col for col in X.columns if col not in categoricas_texto]

# 2. Creación de X_combinado
X_combinado = pd.concat([X[columnas_no_categoricas], X_categoricas_df], axis=1)

# 4. Verificar que todo esté correcto
print(f"\n X_combinado creado")
print(f"   • Filas: {X_combinado.shape[0]}")
print(f"   • Columnas: {X_combinado.shape[1]}")
print(f"   • Tipos: {X_combinado.dtypes.unique()}")
print(f"   • ¿Hay texto? {any(X_combinado.dtypes == 'object')}")
print(f"   • Primera columna: {X_combinado.columns[0]}")
print(f"   • Última columna: {X_combinado.columns[-1]}")


 X_combinado creado
   • Filas: 1470
   • Columnas: 48
   • Tipos: [dtype('int64')]
   • ¿Hay texto? False
   • Primera columna: Age
   • Última columna: MaritalStatus_Single


##### Tratamiento del desbalanceo

1. SPLIT ESTRATIFICADO (stratify=y):
   - De esta manera garantizo que train (80%) y test (20%) mantengan la MISMA proporción de Attrition
   - Train: 16.1% rotación, Test: 16.1% rotación
   - Evita que el modelo no vea suficientes casos de rotación

2. CLASS_WEIGHT='BALANCED' en los modelos:
   - De esta manera lo preparo para que durante el entrenamiento, los errores en la clase minoritaria (Attrition=1) pesen MÁS
   - Así el modelo aprende mejor a predecir la rotación
   - Considero que esta es la alternativa a utilizar SMOTE, ya que es más simple y no crea datos sintéticos

Ambas estrategias se complementan para manejar el desbalance moderado (16.1%).

In [30]:
# Importar train_test_split y preparar datos
from sklearn.model_selection import train_test_split

print(f"X_combinado shape: {X_combinado.shape}")
print(f"y shape: {y.shape}")
print(f"Proporción Attrition en y: {y.mean()*100:.2f}%")
print("  (16.1% rotación, 83.9% no rotación)")

X_combinado shape: (1470, 48)
y shape: (1470,)
Proporción Attrition en y: 16.12%
  (16.1% rotación, 83.9% no rotación)


In [31]:
# Ejecutar split estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X_combinado,  # 48 features ya transformadas
    y,            # Target (Attrition)
    test_size=0.2,            # 20% para test
    stratify=y,               # Mantener misma proporción
    random_state=42           # Para reproducibilidad
)

print(f"\nResultados:")
print(f"• X_train: {X_train.shape}")
print(f"• X_test:  {X_test.shape}")
print(f"• y_train: {y_train.shape}")
print(f"• y_test:  {y_test.shape}")


Resultados:
• X_train: (1176, 48)
• X_test:  (294, 48)
• y_train: (1176,)
• y_test:  (294,)


In [32]:
# Verificar estratificación

print(f"Proporción Attrition original: {y.mean()*100:.2f}%")
print(f"Proporción Attrition en train: {y_train.mean()*100:.2f}%")
print(f"Proporción Attrition en test:  {y_test.mean()*100:.2f}%")

# Verificación exacta
if abs(y.mean() - y_train.mean()) < 0.001 and abs(y.mean() - y_test.mean()) < 0.001:
    print("\n Misma proporción en ambos sets")
else:
    print("\n Con pequeñas diferencias por redondeo, pero aceptable")
print(f"\nCasos de rotación:")
print(f"• Train: {y_train.sum()} de {len(y_train)} ({y_train.mean()*100:.1f}%)")
print(f"• Test:  {y_test.sum()} de {len(y_test)} ({y_test.mean()*100:.1f}%)")

Proporción Attrition original: 16.12%
Proporción Attrition en train: 16.16%
Proporción Attrition en test:  15.99%

 Con pequeñas diferencias por redondeo, pero aceptable

Casos de rotación:
• Train: 190 de 1176 (16.2%)
• Test:  47 de 294 (16.0%)


In [33]:
# Guardado de datos preparados para modelado

import os

# Crear carpeta si no existe
ruta_procesados = "../data/processed"

# Guardar TRAIN (para Notebook 2 - entrenamiento)
X_train.to_csv(os.path.join(ruta_procesados, "X_train.csv"), index=False)
y_train.to_csv(os.path.join(ruta_procesados, "y_train.csv"), index=False)
print(f"✓ X_train guardado: {X_train.shape}")
print(f"✓ y_train guardado: {y_train.shape}")

# Guardar TEST (y no se toca hasta Notebook 3 - en la evaluación final)
X_test.to_csv(os.path.join(ruta_procesados, "X_test.csv"), index=False)
y_test.to_csv(os.path.join(ruta_procesados, "y_test.csv"), index=False)
print(f"✓ X_test guardado: {X_test.shape}")
print(f"✓ y_test guardado: {y_test.shape}")

print(f"\n Archivos guardados en: {ruta_procesados}/")
print("   • X_train.csv, y_train.csv → Para entrenar modelos")
print("   • X_test.csv, y_test.csv   → NO TOCAR hasta evaluación final")

✓ X_train guardado: (1176, 48)
✓ y_train guardado: (1176,)
✓ X_test guardado: (294, 48)
✓ y_test guardado: (294,)

 Archivos guardados en: ../data/processed/
   • X_train.csv, y_train.csv → Para entrenar modelos
   • X_test.csv, y_test.csv   → NO TOCAR hasta evaluación final


In [34]:
'''El Split estratificado me da estos resultados:
• Train:
  - 1176 casos (80%)
  - 48 features
  - 190 casos Attrition=1 (16.2%)

• Test (que no se va a tocar hasta evaluación):
  - 294 casos (20%)
  - 48 features
  - 47 casos Attrition=1 (16.0%)
  '''

'El Split estratificado me da estos resultados:\n• Train:\n  - 1176 casos (80%)\n  - 48 features\n  - 190 casos Attrition=1 (16.2%)\n\n• Test (que no se va a tocar hasta evaluación):\n  - 294 casos (20%)\n  - 48 features\n  - 47 casos Attrition=1 (16.0%)\n  '

In [35]:
'''
El siguiente paso es 
1. Crear pipelines para cada modelo
   • Regresión Logística (con StandardScaler)
   • Random Forest (sin escalado)
   • XGBoost (sin escalado)
2. Entrenar modelos con cross-validation
3. Comparar resultados
'''

'\nEl siguiente paso es \n1. Crear pipelines para cada modelo\n   • Regresión Logística (con StandardScaler)\n   • Random Forest (sin escalado)\n   • XGBoost (sin escalado)\n2. Entrenar modelos con cross-validation\n3. Comparar resultados\n'

In [36]:
# Pipelines por modelo

# Importo lo necesario
from sklearn.preprocessing import StandardScaler # StandardScaler (para escalar variables continuas
from sklearn.linear_model import LogisticRegression # LogisticRegression (modelo lineal)
from sklearn.ensemble import RandomForestClassifier # RandomForestClassifier (árboles ensemble)
from xgboost import XGBClassifier # XGBClassifier (gradient boosting)
from sklearn.pipeline import Pipeline # Pipeline (para encadenar pasos)

In [37]:
#  Identificar variables continuas para escalar

print(f"Variables continuas ({len(continuas)}):")
for var in continuas:
    print(f"  • {var}")
    
print(f"\n Variables que tienen escalas muy diferentes:")
print(f"  • Age: {X_train['Age'].min()}-{X_train['Age'].max()} años")
print(f"  • MonthlyIncome: {X_train['MonthlyIncome'].min()}-{X_train['MonthlyIncome'].max()} €")
print(f"  • DailyRate: {X_train['DailyRate'].min()}-{X_train['DailyRate'].max()}")

print("\nPara Regresión Logística necesitan StandardScaler")
print("Para Random Forest y XGBoost NO necesitan escalado")

Variables continuas (12):
  • Age
  • DailyRate
  • DistanceFromHome
  • HourlyRate
  • MonthlyIncome
  • MonthlyRate
  • PercentSalaryHike
  • TotalWorkingYears
  • YearsAtCompany
  • YearsInCurrentRole
  • YearsSinceLastPromotion
  • YearsWithCurrManager

 Variables que tienen escalas muy diferentes:
  • Age: 18-60 años
  • MonthlyIncome: 1009-19973 €
  • DailyRate: 103-1499

Para Regresión Logística necesitan StandardScaler
Para Random Forest y XGBoost NO necesitan escalado


In [38]:
#  Creación de Pipeline para Regresión Logística (con escalado)

from sklearn.compose import ColumnTransformer

# 1. Identificar índices de columnas continuas en X_train
indices_continuas = [X_train.columns.get_loc(col) for col in continuas if col in X_train.columns]

# 2. Crear ColumnTransformer para escalar SOLO continuas
preprocessor_lr = ColumnTransformer([
    ('scaler', StandardScaler(), indices_continuas)
], remainder='passthrough')  # 'passthrough' = las demás columnas sin cambios

# 3. Crear Pipeline completo
pipeline_lr = Pipeline([
    ('preprocessor', preprocessor_lr), # porque hay que escalar solo continuas
    ('classifier', LogisticRegression(
        class_weight='balanced',  # Manejo desbalance
        random_state=42,
        max_iter=1000  # Asegurar convergencia
    ))
])

print("   ✓ Pipeline creado:")
print("     1. preprocessor: StandardScaler para continuas")
print("     2. classifier: LogisticRegression(class_weight='balanced')")

   ✓ Pipeline creado:
     1. preprocessor: StandardScaler para continuas
     2. classifier: LogisticRegression(class_weight='balanced')


In [39]:
#  Creación de Pipeline para Random Forest (sin escalado)

# 2. Crear Pipeline simple (solo el clasificador)
pipeline_rf = Pipeline([
    ('classifier', RandomForestClassifier(
        class_weight='balanced',  # Manejo desbalance
        random_state=42,
        n_estimators=100  # 100 árboles (valor inicial)
    ))
])

print("   ✓ Pipeline creado:")
print("     • classifier: RandomForestClassifier(class_weight='balanced')")

   ✓ Pipeline creado:
     • classifier: RandomForestClassifier(class_weight='balanced')


In [ ]:
#  Creación de Pipeline para XGBoost (sin escalado)

# 2. Calcular scale_pos_weight para el desbalance
    # scale_pos_weight = número de casos negativos / número de casos positivos
negativos = (y_train == 0).sum()
positivos = (y_train == 1).sum()
scale_pos_weight = negativos / positivos

print(f"   • Casos Attrition=0 (negativos): {negativos}")
print(f"   • Casos Attrition=1 (positivos): {positivos}")
print(f"   • scale_pos_weight = {negativos}/{positivos} = {scale_pos_weight:.2f}")
# equivalente a class_weight='balanced' pero para XGBoost

# 3. Crear Pipeline
pipeline_xgb = Pipeline([
    ('classifier', XGBClassifier(
        scale_pos_weight=scale_pos_weight,  # Manejo desbalance
        random_state=42,
        n_estimators=100,  # 100 árboles (valor inicial)
        eval_metric='logloss',  # Métrica para clasificación binaria
        use_label_encoder=False
    ))
])

print("   ✓ Pipeline creado:")
print(f"     • classifier: XGBClassifier(scale_pos_weight={scale_pos_weight:.2f})")

   • Casos Attrition=0 (negativos): 986
   • Casos Attrition=1 (positivos): 190
   • scale_pos_weight = 986/190 = 5.19
   ✓ Pipeline creado:
     • classifier: XGBClassifier(scale_pos_weight=5.19)


In [26]:
'''scale_pos_weight = 5.19 significa que cada caso positivo (Attrition=1) vale como 5.19 casos negativos'''

'scale_pos_weight = 5.19 significa que cada caso positivo (Attrition=1) vale como 5.19 casos negativos'

In [27]:
print("\nRESUMEN PIPELINES CREADOS:")
print(f"1. Regresión Logística: {len(pipeline_lr.steps)} pasos")
print(f"2. Random Forest: {len(pipeline_rf.steps)} pasos")  
print(f"3. XGBoost: {len(pipeline_xgb.steps)} pasos")

print("\nConfiguración de desbalance:")
print(f"• LogisticRegression: class_weight='balanced'")
print(f"• RandomForest: class_weight='balanced'")
print(f"• XGBoost: scale_pos_weight={scale_pos_weight:.2f}")


RESUMEN PIPELINES CREADOS:
1. Regresión Logística: 2 pasos
2. Random Forest: 1 pasos
3. XGBoost: 1 pasos

Configuración de desbalance:
• LogisticRegression: class_weight='balanced'
• RandomForest: class_weight='balanced'
• XGBoost: scale_pos_weight=5.19


In [ ]:
# Entrenamiento de modelo de Regresión Logística

from sklearn.metrics import accuracy_score
import time

# 1. Entrenar modelo

start_time = time.time()
pipeline_lr.fit(X_train, y_train)
tiempo_lr = time.time() - start_time
print(f"   ✓ Entrenado en {tiempo_lr:.2f} segundos")

# 2. Resultados en train
y_pred_lr_train = pipeline_lr.predict(X_train)
accuracy_train = accuracy_score(y_train, y_pred_lr_train)
print(f"   • Accuracy: {accuracy_train:.3f}")
print(f"   • Attrition=1 predichos: {y_pred_lr_train.sum()} de {len(y_pred_lr_train)}")

'''
Accuracy 0.777 en train, predice correctamente el 77.7% de los casos en entrenamiento
Attrition=1 predice 376 empleados con riesgo de rotación. Hay 190 reales. 
- Sobreestima los casos de rotación por el efecto de class_weight='balanced'. Entiendo que prefiere falsos positivos que falsos negativos
'''


   ✓ Entrenado en 0.05 segundos
   • Accuracy: 0.777
   • Attrition=1 predichos: 376 de 1176


"\nAccuracy 0.777 en train, predice correctamente el 77.7% de los casos en entrenamiento\nAttrition=1 predice 376 empleados con riesgo de rotación. Hay 190 reales. \n- Sobreestima los casos de rotación por el efecto de class_weight='balanced'. Entiendo que prefiere falsos positivos que falsos negativos\n"

In [ ]:
# Entrenamiento de modelo de Random Forest

# 1. Entrenar modelo
start_time = time.time()
pipeline_rf.fit(X_train, y_train)
tiempo_rf = time.time() - start_time
print(f"   ✓ Entrenado en {tiempo_rf:.2f} segundos")

# 2. Resultados en train
y_pred_rf_train = pipeline_rf.predict(X_train)
accuracy_rf_train = accuracy_score(y_train, y_pred_rf_train)
print(f"   • Accuracy: {accuracy_rf_train:.3f}")
print(f"   • Attrition=1 predichos: {y_pred_rf_train.sum()} de {len(y_pred_rf_train)}")

print("\nFeature Importance (primeras 5 más importantes):")
importancias = pipeline_rf.named_steps['classifier'].feature_importances_
# Emparejar nombres de features con importancias
nombres_features = X_train.columns
top_5_idx = importancias.argsort()[-5:][::-1]  # Índices de las 5 más importantes

for i, idx in enumerate(top_5_idx, 1):
    print(f"   {i}. {nombres_features[idx]}: {importancias[idx]:.4f}")

'''
Sobreajuste.
Accuracy: 1.000 → 100% correcto en train
Attrition=1 predichos: 190 ← que coincide con la realidad (190)
Random Forest memorizó los datos en lugar de aprender patrones
'''

   ✓ Entrenado en 0.17 segundos
   • Accuracy: 1.000
   • Attrition=1 predichos: 190 de 1176

Feature Importance (primeras 5 más importantes):
   1. MonthlyIncome: 0.0657
   2. Age: 0.0602
   3. TotalWorkingYears: 0.0544
   4. DailyRate: 0.0513
   5. MonthlyRate: 0.0476


'\nSobreajuste.\nAccuracy: 1.000 → 100% correcto en train\nAttrition=1 predichos: 190 ← que coincide con la realidad (190)\nRandom Forest memorizó los datos en lugar de aprender patrones\n'

In [ ]:
# Entrenamiento del modelo de XGBoost

from sklearn.metrics import accuracy_score
import time

# 1. Entrenar modelo
start_time = time.time()
pipeline_xgb.fit(X_train, y_train)
tiempo_xgb = time.time() - start_time
print(f"   ✓ Entrenado en {tiempo_xgb:.2f} segundos")

# 2. Resultados en train
y_pred_xgb_train = pipeline_xgb.predict(X_train)
accuracy_xgb_train = accuracy_score(y_train, y_pred_xgb_train)
print(f"   • Accuracy: {accuracy_xgb_train:.3f}")
print(f"   • Attrition=1 predichos: {y_pred_xgb_train.sum()} de {len(y_pred_xgb_train)}")

print("\nFeature Importance XGBoost (primeras 5):")
importancias_xgb = pipeline_xgb.named_steps['classifier'].feature_importances_
nombres_features = X_train.columns
top_5_idx_xgb = importancias_xgb.argsort()[-5:][::-1]

for i, idx in enumerate(top_5_idx_xgb, 1):
    print(f"   {i}. {nombres_features[idx]}: {importancias_xgb[idx]:.4f}")

'''
Sobreajuste. 
Accuracy 1.000
Memoriza los datos en lugar de aprender patrones generalizables.
'''

   ✓ Entrenado en 0.11 segundos
   • Accuracy: 1.000
   • Attrition=1 predichos: 190 de 1176

Feature Importance XGBoost (primeras 5):
   1. burnout_risk: 0.0767
   2. JobLevel: 0.0567
   3. OverTime: 0.0513
   4. JobRole_Research Director: 0.0485
   5. StockOptionLevel: 0.0476


c:\Users\usuario\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:199: UserWarning: [20:23:46] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


'\nSobreajuste. \nAccuracy 1.000\nMemoriza los datos en lugar de aprender patrones generalizables.\n'

In [ ]:
# Utilizo cross-validation para estimar el rendimiento del modelo de forma más robusta y reducir la dependencia de una 
# única partición train-test.

from sklearn.model_selection import cross_val_score, StratifiedKFold
import numpy as np

# Configurar CV (5 folds estratificados)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Métricas interesantes
metricas = ['accuracy', 'f1', 'precision', 'recall']
resultados_cv = {}

print("\nEVALUANDO LOS 3 MODELOS CON 5-FOLD CV:")

for nombre, pipeline in [('LogisticRegression', pipeline_lr),
                         ('RandomForest', pipeline_rf),
                         ('XGBoost', pipeline_xgb)]:
    
    print(f"\n• {nombre}:")
    resultados_modelo = {}
    
    for metrica in metricas:
        scores = cross_val_score(pipeline, X_train, y_train, 
                                cv=cv, scoring=metrica)
        resultados_modelo[metrica] = {
            'mean': scores.mean(),
            'std': scores.std(),
            'scores': scores
        }
        
        print(f"  - {metrica:10}: {scores.mean():.3f} (+/- {scores.std():.3f})")
    
    resultados_cv[nombre] = resultados_modelo

print("\n" + "="*60)
print("COMPARACIÓN FINAL (F1 Score - métrica principal)")
print("="*60)

# Ordenar por F1 score
for nombre in sorted(resultados_cv.keys(), 
                    key=lambda x: resultados_cv[x]['f1']['mean'], 
                    reverse=True):
    f1_mean = resultados_cv[nombre]['f1']['mean']
    f1_std = resultados_cv[nombre]['f1']['std']
    acc_mean = resultados_cv[nombre]['accuracy']['mean']
    
    print(f"{nombre:20} → F1: {f1_mean:.3f} (±{f1_std:.3f}) | Accuracy: {acc_mean:.3f}")

'''
1. Regresión Logística:
- F1: 0.486 (mejor)
- Accuracy: 0.747
- Recall: 0.737 (Muy alto. Detecta 73.7% de los que se van)
- Precision: 0.365 (baja - muchos falsos positivos)
--> Interpretación: Prefiere no perder casos reales (alto recall)

2. XGBoost:
- F1: 0.483 (casi igual a Logistic)
- Accuracy: 0.866 (más alto)
- Recall: 0.389 (bajo - pierde muchos casos reales)
- Precision: 0.643 (mejor - menos falsos positivos)

3. Random Forest:
- F1: 0.240 (peor)
- Accuracy: 0.857 (alto pero engañoso)
- Recall: 0.142 (muy bajo - solo detecta 14.2% de rotación)
- Precision: 0.875 (muy alto)
'''


EVALUANDO LOS 3 MODELOS CON 5-FOLD CV:

• LogisticRegression:
  - accuracy  : 0.747 (+/- 0.029)
  - f1        : 0.486 (+/- 0.026)
  - precision : 0.365 (+/- 0.030)
  - recall    : 0.737 (+/- 0.069)

• RandomForest:
  - accuracy  : 0.857 (+/- 0.006)
  - f1        : 0.240 (+/- 0.063)
  - precision : 0.875 (+/- 0.116)
  - recall    : 0.142 (+/- 0.043)

• XGBoost:


c:\Users\usuario\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:199: UserWarning: [20:23:50] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\usuario\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:199: UserWarning: [20:23:50] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\usuario\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:199: UserWarning: [20:23:50] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\usuario\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:199: UserWarning: [20:23:50] WARNING: 

  - accuracy  : 0.866 (+/- 0.016)


c:\Users\usuario\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:199: UserWarning: [20:23:50] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\usuario\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:199: UserWarning: [20:23:50] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\usuario\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:199: UserWarning: [20:23:51] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\usuario\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:199: UserWarning: [20:23:51] WARNING: 

  - f1        : 0.483 (+/- 0.075)


...

  - precision : 0.643 (+/- 0.092)


c:\Users\usuario\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:199: UserWarning: [20:23:51] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\usuario\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:199: UserWarning: [20:23:51] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\usuario\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:199: UserWarning: [20:23:51] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


  - recall    : 0.389 (+/- 0.069)

COMPARACIÓN FINAL (F1 Score - métrica principal)
LogisticRegression   → F1: 0.486 (±0.026) | Accuracy: 0.747
XGBoost              → F1: 0.483 (±0.075) | Accuracy: 0.866
RandomForest         → F1: 0.240 (±0.063) | Accuracy: 0.857


c:\Users\usuario\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:199: UserWarning: [20:23:51] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\usuario\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:199: UserWarning: [20:23:51] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


'\n1. Regresión Logística:\n- F1: 0.486 (mejor)\n- Accuracy: 0.747\n- Recall: 0.737 (¡muy alto! detecta 73.7% de los que se van)\n- Precision: 0.365 (baja - muchos falsos positivos)\n--> Interpretación: Prefiere no perder casos reales (alto recall)\n\n2. XGBoost:\n- F1: 0.483 (casi igual a Logistic)\n- Accuracy: 0.866 (más alto)\n- Recall: 0.389 (bajo - pierde muchos casos reales)\n- Precision: 0.643 (mejor - menos falsos positivos)\n\n3. Random Forest:\n- F1: 0.240 (peor)\n- Accuracy: 0.857 (alto pero engañoso)\n- Recall: 0.142 (muy bajo - solo detecta 14.2% de rotación)\n- Precision: 0.875 (muy alto)\n'

In [ ]:
'''
Los árboles (RF y XGBoost) sobreajustan en train (accuracy 1.000) pero no generalizan bien en cross-validation.
Solución: Optimizar hiperparámetros (GridSearch)
'''
# Optimizar Random Forest (el peor de los tres)

from sklearn.model_selection import GridSearchCV
import warnings
warnings.filterwarnings('ignore')

# 1. Definir parámetros a probar
print("\n1. Definir grid de parámetros para Random Forest:")

param_grid_rf = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [5, 10, None],
    'classifier__min_samples_split': [10, 20],
    'classifier__min_samples_leaf': [5, 10],
    'classifier__max_features': ['sqrt', 'log2']
}

print("   • n_estimators: [50, 100]")
print("   • max_depth: [5, 10, None] (None = sin límite)")
print("   • min_samples_split: [10, 20] (mínimo para dividir nodo)")
print("   • min_samples_leaf: [5, 10] (mínimo en hoja)")
print("   • max_features: ['sqrt', 'log2'] (features por árbol)")

# 2. Configurar GridSearch
print("\n2. Configurar GridSearchCV:")
print("   • Scoring: F1 (métrica principal)")
print("   • CV: 3 folds (para ser rápido)")
print("   • n_jobs: -1 (usar todos los cores)")

grid_rf = GridSearchCV(
    pipeline_rf,
    param_grid_rf,
    scoring='f1',
    cv=3,
    n_jobs=-1,
    verbose=1
)

# 3. Ejecutar GridSearch
print("\n3. Ejecutando GridSearch (puede tardar unos minutos)...")
grid_rf.fit(X_train, y_train)

print("\n4. Mejores parámetros encontrados:")
for param, value in grid_rf.best_params_.items():
    print(f"   • {param}: {value}")

print(f"\n5. Mejor score F1 en CV: {grid_rf.best_score_:.3f}")
print(f"   Mejora respecto a baseline (0.240): {grid_rf.best_score_ - 0.240:.3f}")

# Guardar el mejor modelo
best_rf = grid_rf.best_estimator_


1. Definir grid de parámetros para Random Forest:
   • n_estimators: [50, 100]
   • max_depth: [5, 10, None] (None = sin límite)
   • min_samples_split: [10, 20] (mínimo para dividir nodo)
   • min_samples_leaf: [5, 10] (mínimo en hoja)
   • max_features: ['sqrt', 'log2'] (features por árbol)

2. Configurar GridSearchCV:
   • Scoring: F1 (métrica principal)
   • CV: 3 folds (para ser rápido)
   • n_jobs: -1 (usar todos los cores)

3. Ejecutando GridSearch (puede tardar unos minutos)...
Fitting 3 folds for each of 48 candidates, totalling 144 fits

4. Mejores parámetros encontrados:
   • classifier__max_depth: 5
   • classifier__max_features: log2
   • classifier__min_samples_leaf: 10
   • classifier__min_samples_split: 10
   • classifier__n_estimators: 100

5. Mejor score F1 en CV: 0.510
   Mejora respecto a baseline (0.240): 0.270


In [ ]:
'''
El resultado da los mejores parámetros:
- max_depth: 5 ← Árboles más simples (antes era None = sin límite)
- max_features: log2 ← Usa menos features por árbol
- min_samples_leaf: 10 ← Hojas más grandes
- min_samples_split: 10 ← Más datos para dividir
- n_estimators: 100 ← Mantiene 100 árboles

Hay mejora.
- Baseline F1: 0.240 (malo)
- Optimizado F1: 0.510 (muy bueno)
- Mejora: +0.270 (más del doble)

--> Interpretación:
- Random Forest sobreajustaba mucho (árboles muy profundos sin restricciones).
- Ahora con árboles simples (profundidad 5) y más regularización, generaliza mucho mejor.
'''

'\nEl resultado nos da los mejores parámetros:\n- max_depth: 5 ← ¡Árboles más simples! (antes era None = sin límite)\n- max_features: log2 ← Usa menos features por árbol\n- min_samples_leaf: 10 ← Hojas más grandes\n- min_samples_split: 10 ← Más datos para dividir\n- n_estimators: 100 ← Mantiene 100 árboles\n\nHay mejora.\n- Baseline F1: 0.240 (malo)\n- Optimizado F1: 0.510 (muy bueno)\n- Mejora: +0.270 (más del doble)\n\n--> Interpretación:\n- Random Forest sobreajustaba mucho (árboles muy profundos sin restricciones).\n- Ahora con árboles simples (profundidad 5) y más regularización, generaliza mucho mejor.\n'

In [ ]:
# Segundo, optimizar XGBoost (el segundo mejor)

# 1. Definir parámetros para XGBoost
print("\n1. Definir grid de parámetros para XGBoost:")

param_grid_xgb = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [3, 5, 7],
    'classifier__learning_rate': [0.01, 0.1, 0.2],
    'classifier__subsample': [0.8, 1.0],
    'classifier__colsample_bytree': [0.8, 1.0]
}

print("   • n_estimators: [50, 100]")
print("   • max_depth: [3, 5, 7] (árboles más simples)")
print("   • learning_rate: [0.01, 0.1, 0.2] (tasa de aprendizaje)")
print("   • subsample: [0.8, 1.0] (% de datos por árbol)")
print("   • colsample_bytree: [0.8, 1.0] (% de features por árbol)")

# 2. Configurar GridSearch
print("\n2. Configurar GridSearchCV para XGBoost:")
print("   • Scoring: F1")
print("   • CV: 3 folds")
print("   • n_jobs: -1")

# Eliminar warning de use_label_encoder
from xgboost import XGBClassifier

pipeline_xgb_fixed = Pipeline([
    ('classifier', XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric='logloss',
        use_label_encoder=False
    ))
])

grid_xgb = GridSearchCV(
    pipeline_xgb_fixed,
    param_grid_xgb,
    scoring='f1',
    cv=3,
    n_jobs=-1,
    verbose=1
)

# 3. Ejecutar GridSearch
print("\n3. Ejecutando GridSearch para XGBoost...")
grid_xgb.fit(X_train, y_train)

print("\n4. Mejores parámetros encontrados:")
for param, value in grid_xgb.best_params_.items():
    print(f"   • {param}: {value}")

print(f"\n5. Mejor score F1 en CV: {grid_xgb.best_score_:.3f}")
print(f"   Mejora respecto a baseline (0.483): {grid_xgb.best_score_ - 0.483:.3f}")

# Guardar mejor modelo
best_xgb = grid_xgb.best_estimator_


1. Definir grid de parámetros para XGBoost:
   • n_estimators: [50, 100]
   • max_depth: [3, 5, 7] (árboles más simples)
   • learning_rate: [0.01, 0.1, 0.2] (tasa de aprendizaje)
   • subsample: [0.8, 1.0] (% de datos por árbol)
   • colsample_bytree: [0.8, 1.0] (% de features por árbol)

2. Configurar GridSearchCV para XGBoost:
   • Scoring: F1
   • CV: 3 folds
   • n_jobs: -1

3. Ejecutando GridSearch para XGBoost...
Fitting 3 folds for each of 72 candidates, totalling 216 fits

4. Mejores parámetros encontrados:
   • classifier__colsample_bytree: 0.8
   • classifier__learning_rate: 0.01
   • classifier__max_depth: 5
   • classifier__n_estimators: 100
   • classifier__subsample: 0.8

5. Mejor score F1 en CV: 0.547
   Mejora respecto a baseline (0.483): 0.064


In [ ]:
#  Tercero optimizar Logistic Regression

from sklearn.model_selection import GridSearchCV

param_grid_lr = {
    'classifier__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'classifier__penalty': ['l1', 'l2'],
    'classifier__solver': ['liblinear', 'saga'],
    'classifier__class_weight': ['balanced', None]
}

grid_lr = GridSearchCV(
    pipeline_lr,
    param_grid_lr,
    scoring='f1',
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid_lr.fit(X_train, y_train)
best_lr = grid_lr.best_estimator_
print(f"LR optimizado - Mejor F1: {grid_lr.best_score_:.3f}")

Fitting 3 folds for each of 48 candidates, totalling 144 fits
LR optimizado - Mejor F1: 0.544


In [36]:
# parametros optimos de Logistic Regression

for param, value in grid_lr.best_params_.items():
    print(f"  • {param}: {value}")

# ¿Qué regularización encontró?
# C alto = poca regularización (más complejo)
# C bajo = mucha regularización (más simple)
# penalty='l1' o 'l2'?

  • classifier__C: 1
  • classifier__class_weight: None
  • classifier__penalty: l1
  • classifier__solver: saga


In [ ]:
# Comparación sistemática de los 3 modelos optimizados

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Lista de modelos optimizados
modelos_optimizados = {
    'Logistic_Regression': best_lr,
    'Random_Forest': best_rf,
    'XGBoost': best_xgb
}

print("COMPARACIÓN DE MODELOS OPTIMIZADOS EN TRAIN (usando predicciones)")
print("="*70)

resultados_comparacion = {}

for nombre, modelo in modelos_optimizados.items():
    print(f"\n📊 {nombre}:")
    print("-" * 40)
    
    # 1. Predicciones en train
    y_pred = modelo.predict(X_train)
    y_pred_proba = modelo.predict_proba(X_train)[:, 1] if hasattr(modelo, 'predict_proba') else None
    
    # 2. Métricas básicas
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    
    resultados = {
        'Accuracy': accuracy_score(y_train, y_pred),
        'Precision': precision_score(y_train, y_pred),
        'Recall': recall_score(y_train, y_pred),
        'F1': f1_score(y_train, y_pred)
    }
    
    # 3. AUC-ROC si tiene probabilidades
    if y_pred_proba is not None:
        resultados['AUC-ROC'] = roc_auc_score(y_train, y_pred_proba)
    
    # 4. Guardar resultados
    resultados_comparacion[nombre] = resultados
    
    # 5. Imprimir
    for metrica, valor in resultados.items():
        print(f"  • {metrica:12}: {valor:.3f}")
    
    # 6. Matriz de confusión (solo números)
    cm = confusion_matrix(y_train, y_pred)
    print(f"  • Matriz Confusión: TP={cm[1,1]}, FP={cm[0,1]}, FN={cm[1,0]}, TN={cm[0,0]}")

COMPARACIÓN DE MODELOS OPTIMIZADOS EN TRAIN (usando predicciones)

📊 Logistic_Regression:
----------------------------------------
  • Accuracy    : 0.901
  • Precision   : 0.841
  • Recall      : 0.474
  • F1          : 0.606
  • AUC-ROC     : 0.872
  • Matriz Confusión: TP=90, FP=17, FN=100, TN=969

📊 Random_Forest:
----------------------------------------
  • Accuracy    : 0.886
  • Precision   : 0.621
  • Recall      : 0.758
  • F1          : 0.682
  • AUC-ROC     : 0.913
  • Matriz Confusión: TP=144, FP=88, FN=46, TN=898

📊 XGBoost:
----------------------------------------
  • Accuracy    : 0.940
  • Precision   : 0.777
  • Recall      : 0.879
  • F1          : 0.825
  • AUC-ROC     : 0.978
  • Matriz Confusión: TP=167, FP=48, FN=23, TN=938


#### Justificación de modelo elegido

XGBoost con Recall del 88%.

Tras evaluar tres modelos de machine learning para predecir la rotación de personal, he seleccionado XGBoost como modelo final. Esta decisión se basa en un análisis cuidadoso del equilibrio entre detección temprana y eficiencia operativa.


Comparación clave de resultados:

--> Regresión Logística tiene la mayor precisión (84%), pero un recall muy bajo (47%). Esto significa  que aunque casi siempre acierta cuando dice que alguien se va, se le escapan más de la mitad de los casos reales (100 falsos negativos). Para RRHH, esto es inaceptable: significa perder talento sin oportunidad de intervenir.

--> Random Forest muestra un mejor equilibrio con 76% de recall, pero aún presenta 46 casos no detectados y una precisión menor (62%).

--> XGBoost ofrece el mejor equilibrio global: detecta el 88% de los casos reales (solo 23 no detectados) manteniendo una precisión del 78%.

La Filosofía detrás de mi Elección:

En retención de talento, es mejor aplicar el principio: "Es mejor una falsa alarma que un incendio no detectado".

* Falso Positivo (48 casos): Decimos que alguien se va pero se queda → tiene un coste moderado (tiempo de RRHH en intervención preventiva)
* Falso Negativo (23 casos): Decimos que alguien se queda pero se va → tiene un coste muy alto (pérdida de talento, conocimiento y costes de reemplazo)

XGBoost minimiza precisamente el error más costoso: solo 23 falsos negativos frente a 100 de la Regresión Logística.

Traducción a Impacto Real en la Empresa: con nuestra población de entrenamiento (1,176 empleados):

Con XGBoost, el departamento de RRHH:
- Tendría oportunidad de retener a 167 empleados que realmente planean irse
- Solo perdería sin intento a 23 empleados (vs 100 con otro modelo)
- Gestionaría intervenciones con 215 empleados, de los cuales 167 realmente las necesitan

El Compromiso Óptimo entre Sensibilidad y Especificidad
- El AUC-ROC de 0.978 (casi perfecto) confirma que XGBoost discrimina excelentemente entre quienes se van y quienes se quedan. 
- El recall del 88% representa el punto óptimo donde:
    -> Detectamos la inmensa mayoría de los casos reales
    -> Mantenemos una precisión que permite una intervención eficiente
    -> No saturamos a RRHH con demasiadas falsas alarmas

Conclusión para Implementación
Hemos elegido XGBoost porque prioriza la prevención sobre la precisión pura. En el contexto de recursos humanos, donde el coste de perder talento es muy superior al coste de intervenciones preventivas, esta orientación hacia alta detección (recall) es estratégicamente correcta.
Los 3 números que realmente importan a RRHH:
* ¿A cuánta gente detectamos antes de que se vaya? (Recall)
* ¿Cuántas "falsas alarmas" manejamos? (Precision → carga de trabajo)
* ¿Qué casos se nos escapan? (Falsos Negativos → impacto real)

Este modelo será nuestra herramienta de alerta temprana, permitiendo al departamento de RRHH intervenir proactivamente en el 88% de los casos de rotación potencial, optimizando así la retención del talento y reduciendo costes organizativos.

In [ ]:
# Guardar el modelo XGBoost optimizado

import joblib
import pickle
import os

# 2. Crear carpeta para modelos 
model_dir = "../models"
if not os.path.exists(model_dir):
    os.makedirs(model_dir)
    print(f"✅ Carpeta creada: {model_dir}")

# 3. Guardar el modelo XGBoost optimizado
model_path = os.path.join(model_dir, "xgboost_model.pkl")
joblib.dump(best_xgb, model_path)
print(f"✅ Modelo XGBoost guardado: {model_path}")

# 4. Guardar Onehot encoder
encoder_path = os.path.join(model_dir, "onehot_encoder.pkl")
joblib.dump(onehot, encoder_path)
print(f"✅ OneHotEncoder guardado: {encoder_path}")

# 5. Guardar información de columnas
column_info = {
    'X_columns': list(X_train.columns),  # Todas las columnas usadas
    'categorical_columns': categoricas_texto,  # Columnas categóricas
    'target': 'Attrition',
    'data_shape': X_train.shape
}

info_path = os.path.join(model_dir, "column_info.pkl")
with open(info_path, 'wb') as f:
    pickle.dump(column_info, f)
print(f"✅ Información de columnas guardada: {info_path}")

# 6. RESUMEN

print(f"""
1. Modelo principal:    xgboost_model.pkl
2. Transformador:       onehot_encoder.pkl  
3. Información:         column_info.pkl
""")


✅ Modelo XGBoost guardado: ../models\xgboost_model.pkl
✅ OneHotEncoder guardado: ../models\onehot_encoder.pkl
✅ Información de columnas guardada: ../models\column_info.pkl

1. 📊 Modelo principal:    xgboost_model.pkl
2. 🔧 Transformador:       onehot_encoder.pkl  
3. 📝 Información:         column_info.pkl



### Evaluación del modelo: métricas utilizadas y justificación

La evaluación de los modelos se ha diseñado teniendo en cuenta tanto las características del dataset como el impacto real de los errores en un contexto de recursos humanos. Dado el fuerte desbalance existente (≈16% de rotación) y el alto coste asociado a no detectar una baja real, métricas globales como la accuracy no resultan adecuadas como criterio principal de evaluación. Por este motivo, se han priorizado métricas centradas en la clase minoritaria (Attrition = 1) y en el equilibrio entre capacidad de detección y eficiencia operativa, con el objetivo de seleccionar un modelo útil y aplicable en un entorno real de toma de decisiones.

- **Recall**
Mide qué porcentaje de los empleados que realmente abandonan la empresa es detectado por el modelo. En este problema, valores bajos implican perder talento sin posibilidad de intervención, por lo que esta métrica se considera prioritaria. El modelo XGBoost alcanza un recall del 88%, lo que significa que detecta casi 9 de cada 10 bajas reales.

- **Precision**
Indica qué proporción de las alertas generadas por el modelo corresponde a bajas reales. Esta métrica permite evaluar la carga operativa para el departamento de RRHH, ya que valores muy bajos implicarían un número elevado de intervenciones innecesarias. XGBoost presenta una precision del 78%, considerada adecuada para mantener un equilibrio entre detección y eficiencia.

- **F1-score**
Es una métrica de equilibrio entre recall y precision, especialmente útil en problemas con clases desbalanceadas. Se ha utilizado para comparar los modelos de forma justa, evitando soluciones extremas que priorizan una métrica a costa de la otra. XGBoost obtiene el mejor F1-score, reflejando el mejor compromiso global entre ambas.

- **Accuracy**
Mide el porcentaje total de predicciones correctas. En este contexto, se ha utilizado únicamente como referencia general, ya que un valor alto puede estar dominado por la correcta clasificación de la clase mayoritaria y no reflejar la capacidad real de detección de la rotación.

- **AUC-ROC**
Evalúa la capacidad del modelo para discriminar entre empleados con y sin riesgo de rotación. Valores cercanos a 1 indican una excelente separación entre clases. El AUC-ROC obtenido por XGBoost (0.978) confirma su alta capacidad discriminativa, aunque esta métrica se considera complementaria y no decisiva.

***Conclusión***

Los resultados muestran que XGBoost ofrece el mejor equilibrio entre detección temprana de la rotación y eficiencia operativa. Su alto recall reduce de forma significativa los falsos negativos, mientras que su precision permite gestionar las intervenciones de RRHH sin generar una carga excesiva. Por este motivo, se selecciona como modelo final y como base para la posterior evaluación en el conjunto de test y el ajuste del umbral de decisión.